# 03 — Variabilidade em escala: curvas de luz do ZTF

Workshop SAB — AI-Scope / ADSS

**Pergunta científica:** como passar de uma seleção espacial para comportamento temporal sem baixar uma base inteira de séries temporais?

`public.ztf_dr22` armazena `hmjd`, `mag`, `magerr` e `catflags` como arrays. O objetivo é selecionar fontes no servidor e transformar apenas as séries retornadas em uma representação adequada para análise temporal.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adss

BASE_URL = "https://ai-scope.cbpf.br/"
USERNAME = os.getenv("AI_SCOPE_USERNAME")
PASSWORD = os.getenv("AI_SCOPE_PASSWORD")

if not USERNAME or not PASSWORD:
    raise RuntimeError("Defina AI_SCOPE_USERNAME e AI_SCOPE_PASSWORD no ambiente.")

cl = adss.ADSSClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)

In [ ]:
CENTER_RA = 150.0
CENTER_DEC = 2.0
RADIUS_DEG = 0.1

query = f"""
SELECT TOP 500
    id, ra, dec, fieldid, filterid, nepochs,
    hmjd, mag, magerr, catflags
FROM public.ztf_dr22
WHERE cone(ra, dec, {CENTER_RA}, {CENTER_DEC}, {RADIUS_DEG})
  AND nepochs >= 30
"""
ztf = cl.query_and_wait(query_text=query, mode="astroql").data
ztf[["id", "ra", "dec", "filterid", "nepochs"]].head()

In [ ]:
# Escolha uma fonte bem amostrada
row = ztf.sort_values("nepochs", ascending=False).iloc[0]
t = np.asarray(row.hmjd, dtype=float)
m = np.asarray(row.mag, dtype=float)
e = np.asarray(row.magerr, dtype=float)
valid = np.isfinite(t) & np.isfinite(m) & np.isfinite(e)

fig, ax = plt.subplots(figsize=(9,4))
ax.errorbar(t[valid], m[valid], yerr=e[valid], fmt=".", ms=3, alpha=0.7)
ax.invert_yaxis()
ax.set(xlabel="HMJD", ylabel="magnitude", title=f"ZTF source {row.id}")
plt.show()

In [ ]:
from astropy.timeseries import LombScargle

frequency, power = LombScargle(t[valid], m[valid], e[valid]).autopower(
    minimum_frequency=1/100,
    maximum_frequency=10,
)
best_frequency = frequency[np.argmax(power)]
best_period = 1 / best_frequency
print(f"Candidate period: {best_period:.5f} d")

phase = (t[valid] % best_period) / best_period
fig, ax = plt.subplots(figsize=(7,4))
ax.errorbar(phase, m[valid], yerr=e[valid], fmt=".", ms=3, alpha=0.7)
ax.invert_yaxis()
ax.set(xlabel="phase", ylabel="magnitude", title="Phase-folded light curve")
plt.show()

### Extensão inovadora
Em vez de classificar cada curva diretamente, construa *features* (amplitude robusta, período, entropia, assimetria, dispersão) para centenas de objetos e procure *outliers*. O objetivo passa a ser **descoberta de comportamento raro**, e não reprodução de classes conhecidas.

In [ ]:
def variability_features(row):
    m = np.asarray(row.mag, dtype=float)
    m = m[np.isfinite(m)]
    if len(m) < 10:
        return pd.Series({"amp90": np.nan, "std": np.nan, "mad": np.nan})
    med = np.median(m)
    return pd.Series({
        "amp90": np.percentile(m, 95) - np.percentile(m, 5),
        "std": np.std(m),
        "mad": np.median(np.abs(m-med)),
    })

features = pd.concat([ztf[["id","ra","dec","nepochs"]], ztf.apply(variability_features, axis=1)], axis=1)
features.sort_values("amp90", ascending=False).head(15)